In [1]:
####本代码用于正式实验的全部正负样本分割
import ee
ee.Authenticate()
ee.Initialize(project='the-second-project-508112')

import os
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [5]:
# ---------------- 1) 参数与核心函数 ----------------
BUFFER_SIZE = 3000

ABS_THRESHOLD_DB = -21
DB_THRESHOLD_SHIFT = 3
BRIGHT_THRESHOLD = -15

# 形态学处理
MORPH_RADIUS = 1 #半径为1像元，形态学方形核大致是：半径约 10 m；边长约 30 m；覆盖约 (3×3) 个 10 m 像元
MIN_AREA_KM2 = 0.02

# lee滤波参数
LEE_KERNEL_RADIUS = 1
LEE_ENL = 4

# 文件夹存储路径名：'pos' 或 'neg'
SAMPLE_TAG = 'pos'
# 资产路径
address = 'projects/the-second-project-508112/assets/pos2014'
fc = ee.FeatureCollection(address)

#尺度设置
FEATURE_SCALE = 10
TEXTURE_SCALE = 10
TEXTURE_LEVELS = 32
BACKGROUND_RING_M = 60
MIN_OBJECT_PIXELS = 9
EPS = 1e-6
PI = 3.141592653589793
SAFE_FILL = -999
# 标签字段：正样本保留 GEE asset 的 Subcategor，负样本保留 Category。
SAMPLE_LABEL_PROPERTY = 'Subcategor' if SAMPLE_TAG == 'pos' else 'Category'

# 同日多景拼接后的 SAR 有效覆盖率下限。小于该值即标记为 SAR 数据缺失，不参与分割。
SAR_COVERAGE_MIN = 0.8


def safe_name(text):
    text = str(text)
    return ''.join(ch if ch.isalnum() or ch in ['_', '-'] else '_' for ch in text)


def asset_basename(asset_path):
    return safe_name(str(asset_path).split('/')[-1])


def short_scene_name(scene_id):
    """
    eg：
    S1A_IW_GRDH_1SDV_20240130T003132_20240130T003201_052329_0653C9_9C82
    -> 0653C9_9C82

    只保留 scene_id 最后两段，文件名更短。
    如果 scene_id 格式不符合预期，则退回 safe_name(scene_id)。
    """
    parts = safe_name(scene_id).split('_')
    if len(parts) >= 2:
        return '_'.join(parts[-2:])
    return safe_name(scene_id)


def build_export_names(scene_id, sar_time, scene_index=None, prefix='region_scene'):
    short_id = short_scene_name(scene_id)
    date_part = str(sar_time)[:10].replace('-', '')
    if scene_index is None:
        return f'{prefix}_{date_part}_{short_id}'
    return f'{prefix}_{scene_index:03d}_{date_part}_{short_id}'


def _safe_num(x, default=SAFE_FILL):
    # 用 firstNonNull 兜底：x 为 null 时返回 default。
    # 之前用 ee.Algorithms.If(IsEqual(x, None), ...)，但 IsEqual(null, null) 在 GEE 中可能返回
    # null，导致 If 走 false 分支把 null 透传给 ee.Image.constant() 而报错，这里改成更稳的写法。
    return ee.Number(ee.List([x, default]).reduce(ee.Reducer.firstNonNull()))


def _safe_div(a, b, default=SAFE_FILL):
    a = ee.Number(a)
    b = ee.Number(b)
    return ee.Number(ee.Algorithms.If(b.abs().gt(EPS), a.divide(b), default))


def _safe_sqrt(x):
    return ee.Number(x).max(0).sqrt()


# ---------------- Lee滤波 ----------------
def lee_filter(img_db):
    """
    简化 Lee 滤波：功率域（线性域）局部统计 MMSE，返回 dB。
    对整个square进行lee滤波，输出波段名为VV
    """
    img_db = img_db.toFloat()
    img_power = ee.Image.constant(10).pow(img_db.divide(10))   # dB -> power

    kernel = ee.Kernel.square(radius=LEE_KERNEL_RADIUS, units='pixels')

    # 局部一、二阶矩
    mean_power = img_power.reduceNeighborhood(ee.Reducer.mean(), kernel)
    mean_sq    = img_power.pow(2).reduceNeighborhood(ee.Reducer.mean(), kernel)
    var_power  = mean_sq.subtract(mean_power.pow(2)).max(0)     # 总体方差

    # 斑点噪声方差（乘性模型，强度图）
    noise_var  = mean_power.pow(2).divide(LEE_ENL)
    signal_var = var_power.subtract(noise_var).max(0)

    # Lee 权重 k
    weights = signal_var.divide(var_power.max(1e-12)).clamp(0, 1)

    filtered_power = mean_power.add(
        weights.multiply(img_power.subtract(mean_power))
    ).max(1e-12)

    return filtered_power.log10().multiply(10).rename('VV')


# ---------------- 风速及风向 sin 特征 ----------------
def _wind_features(era5_img, region, scale=1000):
    """一次区域统计输出窗口平均风速和风向单位向量的 sin 分量。"""
    u_img = era5_img.select('u_component_of_wind_10m')
    v_img = era5_img.select('v_component_of_wind_10m')
    speed_img = u_img.pow(2).add(v_img.pow(2)).sqrt().rename('wind_speed')

    # sin(theta) = v / speed；speed 极小时以 1e-6 防止除零。
    # 风向 sin 只在此处计算一次，随后通过 wind_dict 写入最终 Feature。
    wind_dir_sin_img = v_img.divide(speed_img.max(1e-6)).rename('wind_dir_sin')

    wind_stats = speed_img.addBands(wind_dir_sin_img).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        maxPixels=1e6
    )

    return ee.Dictionary({
        'wind_mean': _safe_num(wind_stats.get('wind_speed')),
        'wind_dir_sin': _safe_num(wind_stats.get('wind_dir_sin'), 0)
    })


# ---------------- 单窗口阈值分割 ----------------
def adaptive_threshold_window(sea_img, region, scale=10):
    local_dict = sea_img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        maxPixels=1e8,
        tileScale=8
    )
    # 先直接算均值
    local_mean = _safe_num(local_dict.get('VV'), SAFE_FILL)
    local_mean_img = ee.Image.constant(local_mean).rename('VV')

    # 将亮像元用均值1替换
    bright_mask = sea_img.gt(BRIGHT_THRESHOLD)
    img_suppressed = sea_img.where(bright_mask, local_mean_img)

    local_dict2 = img_suppressed.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=scale,
        maxPixels=1e8,
        tileScale=8
    )
    # 计算替换亮像元后的均值2
    local_mean2 = _safe_num(local_dict2.get('VV'), SAFE_FILL)
    local_mean_img2 = ee.Image.constant(local_mean2).rename('VV')

    # 筛选像元，con1：和均值相差<db阈值(3)，con2：<abs阈值(-21)
    condition1 = img_suppressed.lt(local_mean_img2.subtract(DB_THRESHOLD_SHIFT))
    condition2 = img_suppressed.lt(ABS_THRESHOLD_DB)
    oil_mask_raw = condition1.And(condition2).rename('oil_mask')

    kernel = ee.Kernel.square(MORPH_RADIUS)
    oil_mask_morph = oil_mask_raw.focal_max(kernel=kernel).focal_min(kernel=kernel)

    # 过滤掉面积 < MIN_AREA_KM2 的小斑块，只保留大斑块。
    patch_vectors = oil_mask_morph.selfMask().reduceToVectors(
        geometry=region, scale=scale, geometryType='polygon', eightConnected=True,
        labelProperty='label', maxPixels=1e8, tileScale=8
    )
    patch_vectors = patch_vectors.map(
        lambda f: ee.Feature(f).set('patch_area_km2', ee.Feature(f).geometry().area(1).divide(1e6))
    )
    big_patches = patch_vectors.filter(ee.Filter.gte('patch_area_km2', MIN_AREA_KM2))
    oil_mask_final = ee.Image(0).byte().paint(big_patches, 1).And(oil_mask_morph).unmask(0).rename('oil_mask')


    return {
        'oil_mask': oil_mask_final,
        'img_suppressed': img_suppressed.rename('VV'),
        'local_mean_before': local_mean,
        'local_mean_after': local_mean2
    }


# ---------------- 窗口级特征构建 ----------------
def build_window_feature(feature, region, raw_img, filtered_img, oil_mask, wind_mean, scene_id, instrument_mode, sar_time, local_mean_before, local_mean_after):
    # oil_area_m2 是窗口内全部通过面积筛选的连通暗斑总面积。
    oil_area_m2 = _safe_num(
        ee.Image.pixelArea().rename('area').updateMask(oil_mask.eq(1)).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=region, scale=FEATURE_SCALE,
            maxPixels=1e8, tileScale=8
        ).get('area'), 0
    )

    # 窗口面积直接由 region 的平面面积获得；oil_ratio 因而是总暗斑面积占窗口面积的比例。
    window_area_m2 = _safe_num(region.area(1), 0)
    oil_ratio = _safe_div(oil_area_m2, window_area_m2, 0)
    # 原始/Lee 滤波 VV 均只保留均值与标准差，不再计算未输出的极值。
    vv_stats = raw_img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
        geometry=region, scale=FEATURE_SCALE, maxPixels=1e8, tileScale=8
    )
    vv_filtered_stats = filtered_img.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
        geometry=region, scale=FEATURE_SCALE, maxPixels=1e8, tileScale=8
    )

    return ee.Feature(region).copyProperties(feature).set({
        SAMPLE_LABEL_PROPERTY: feature.get(SAMPLE_LABEL_PROPERTY),
        'scene_id': scene_id,
        'instrument_mode': instrument_mode,
        'sar_time': sar_time.format('YYYY-MM-dd HH:mm:ss'),
        'wind_mean': wind_mean,
        'local_mean_before': local_mean_before,
        'local_mean_after': local_mean_after,
        'vv_mean': _safe_num(vv_stats.get('VV_mean')),
        'vv_stdDev': _safe_num(vv_stats.get('VV_stdDev')),
        'vv_filtered_mean': _safe_num(vv_filtered_stats.get('VV_mean')),
        'vv_filtered_stdDev': _safe_num(vv_filtered_stats.get('VV_stdDev')),
        'oil_area_m2': oil_area_m2,
        'oil_ratio': oil_ratio
    })


# ---------------- 全部连通暗斑联合特征 ----------------
def _multi_dark_spot_features(oil_mask, region):
    """对窗口内全部连通暗斑一次矢量化，并聚合几何、形状和直线度特征。"""
    patches = oil_mask.selfMask().reduceToVectors(
        geometry=region, scale=FEATURE_SCALE, geometryType='polygon',
        eightConnected=True, labelProperty='label', maxPixels=1e8, tileScale=8
    )

    def add_patch_geometry(f):
        geom = ee.Feature(f).geometry()
        area_m2 = geom.area(1)
        perimeter_m = geom.perimeter(1)
        bounds = ee.List(geom.bounds(1).coordinates().get(0))
        p0 = ee.List(bounds.get(0))
        p1 = ee.List(bounds.get(1))
        p2 = ee.List(bounds.get(2))

        # 外接矩形的经纬度跨度换算为近似长度/宽度；L/W 越大说明斑块越狭长。
        width_m = ee.Number(p1.get(0)).subtract(ee.Number(p0.get(0))).abs().multiply(111320)
        height_m = ee.Number(p2.get(1)).subtract(ee.Number(p1.get(1))).abs().multiply(111320)
        lwr = _safe_div(width_m.max(height_m), width_m.min(height_m), 0)

        # 4πA/P²：圆形为 1，形状越狭长或边界越曲折，数值越接近 0。
        compactness = _safe_div(ee.Number(4).multiply(PI).multiply(area_m2), perimeter_m.pow(2), 0)

        # 直线度：对斑块内像元坐标做正交线性拟合。
        # λ1、λ2 为二维坐标协方差的主/次方向方差，R²=λ1/(λ1+λ2)。
        # 接近 1 表示像元沿一条拟合直线高度集中；不依赖直线斜率，竖直斑块同样适用。
        patch_mask = ee.Image(0).byte().paint(geom, 1).selfMask()
        # 使用 SAR 栅格投影下的像元坐标，避免经纬度坐标在不同纬度具有不同尺度。
        coord_array = ee.Image.pixelCoordinates(oil_mask.projection()).toArray().updateMask(patch_mask)
        cov_dict = coord_array.reduceRegion(
            reducer=ee.Reducer.centeredCovariance(), geometry=geom, scale=FEATURE_SCALE,
            maxPixels=1e8, tileScale=8
        )
        # 极小或退化斑块可能无法返回协方差数组，此时使用零矩阵使直线度安全回退为 0。

        
        default_cov = ee.Array([[0, 0], [0, 0]])
        cov = ee.Array(ee.List([
            cov_dict.get('array'),
            default_cov
        ]).reduce(ee.Reducer.firstNonNull()))

        cxx = ee.Number(cov.get([0, 0]))
        cxy = ee.Number(cov.get([0, 1]))
        cyy = ee.Number(cov.get([1, 1]))
        trace = cxx.add(cyy).max(0)
        discriminant = cxx.subtract(cyy).pow(2).add(cxy.pow(2).multiply(4)).max(0).sqrt()
        lambda1 = trace.add(discriminant).divide(2)
        straightness_r2 = _safe_div(lambda1, trace, 0)

        return ee.Feature(f).set({
            'patch_area_m2': area_m2,
            'patch_lwr': lwr,
            'patch_compactness': compactness,
            'patch_straightness_r2': straightness_r2
        })

    patches = patches.map(add_patch_geometry)
    count = patches.size()
    area_stats = ee.Dictionary(patches.aggregate_stats('patch_area_m2'))
    lwr_stats = ee.Dictionary(patches.aggregate_stats('patch_lwr'))
    total_area = _safe_num(patches.aggregate_sum('patch_area_m2'), 0)

    weighted = patches.map(
        lambda f: ee.Feature(f).set({
            'compactness_x_area': ee.Number(ee.Feature(f).get('patch_compactness')).multiply(ee.Number(ee.Feature(f).get('patch_area_m2'))),
            'straightness_x_area': ee.Number(ee.Feature(f).get('patch_straightness_r2')).multiply(ee.Number(ee.Feature(f).get('patch_area_m2')))
        })
    )

    return ee.Dictionary({
        'dark_spot_count': count,
        'dark_spot_area_mean_m2': _safe_num(area_stats.get('mean'), 0),
        'dark_spot_area_std_m2': _safe_num(area_stats.get('sample_sd'), 0),
        'dark_spot_lwr_max': _safe_num(lwr_stats.get('max'), 0),
        'dark_spot_lwr_mean': _safe_num(lwr_stats.get('mean'), 0),
        'dark_spot_compactness_area_weighted': _safe_div(_safe_num(weighted.aggregate_sum('compactness_x_area'), 0), total_area, 0),
        'dark_spot_straightness_r2_area_weighted': _safe_div(_safe_num(weighted.aggregate_sum('straightness_x_area'), 0), total_area, 0)
    })


# ---------------- 全部暗斑的强度、背景与纹理特征 ----------------
def add_object_features(feature, region, raw_img, oil_mask):
    """在全部保留暗斑的并集上计算联合几何、强度、背景对比和纹理特征。"""
    obj = oil_mask.eq(1).selfMask()
    object_pixel_count = _safe_num(
        oil_mask.eq(1).reduceRegion(
            reducer=ee.Reducer.sum(), geometry=region, scale=FEATURE_SCALE,
            maxPixels=1e8, tileScale=8
        ).get('oil_mask'), 0
    )
    valid_object = object_pixel_count.gte(MIN_OBJECT_PIXELS)

    def when_valid_object():
        # 全部暗斑的联合几何特征：面积/形态统计来自每一个连通斑块后再聚合。
        dark_spot_dict = _multi_dark_spot_features(oil_mask, region)
        obj_geom = obj.reduceToVectors(
            geometry=region, scale=FEATURE_SCALE, geometryType='polygon',
            eightConnected=True, labelProperty='label', maxPixels=1e8, tileScale=8
        ).geometry()

        # 全部暗斑并集的原始 VV 统计；背景环围绕全部斑块，空背景时回退到整窗非暗斑区域。
        obj_stats = raw_img.updateMask(obj).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True)
                                     .combine(ee.Reducer.minMax(), '', True),
            geometry=region, scale=FEATURE_SCALE, maxPixels=1e8, tileScale=8
        )
        outer_ring = obj_geom.buffer(BACKGROUND_RING_M).difference(obj_geom, 1)
        bg_mask = ee.Image.constant(1).clip(region).updateMask(obj.Not())
        bg_stats = raw_img.updateMask(bg_mask).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
            geometry=outer_ring, scale=FEATURE_SCALE, maxPixels=1e8, tileScale=8
        )
        fallback_bg_stats = raw_img.updateMask(bg_mask).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), '', True),
            geometry=region, scale=FEATURE_SCALE, maxPixels=1e8, tileScale=8
        )
        mu_sce = _safe_num(ee.List([bg_stats.get('VV_mean'), fallback_bg_stats.get('VV_mean')]).reduce(ee.Reducer.firstNonNull()))
        sigma_sce = _safe_num(ee.List([bg_stats.get('VV_stdDev'), fallback_bg_stats.get('VV_stdDev')]).reduce(ee.Reducer.firstNonNull()))
        mu_obj = _safe_num(obj_stats.get('VV_mean'))
        sigma_obj = _safe_num(obj_stats.get('VV_stdDev'))
        min_obj = _safe_num(obj_stats.get('VV_min'))
        max_obj = _safe_num(obj_stats.get('VV_max'))
        intensity_ratio = _safe_div(mu_obj, mu_sce, 0)
        isdr = _safe_div(sigma_obj, sigma_sce, 0)
        # isri=对象均值/对象标准差，用于描述联合暗斑内部散射的相对离散程度。
        isri = _safe_div(mu_obj, sigma_obj, 0)

        # 在全部暗斑并集内汇总 GLCM 纹理均值。
        texture_img = raw_img.clamp(-35, 0).unitScale(-35, 0).multiply(TEXTURE_LEVELS - 1).toInt().rename('VV')
        tex_stats = texture_img.glcmTexture(size=1).updateMask(obj).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=region, scale=TEXTURE_SCALE,
            maxPixels=1e8, tileScale=8
        )
        texture_dict = ee.Dictionary({
            'asm': _safe_num(tex_stats.get('VV_asm')),
            'contrast': _safe_num(tex_stats.get('VV_contrast')),
            'corr': _safe_num(tex_stats.get('VV_corr')),
            'var': _safe_num(tex_stats.get('VV_var')),
            'idm': _safe_num(tex_stats.get('VV_idm')),
            'savg': _safe_num(tex_stats.get('VV_savg')),
            'svar': _safe_num(tex_stats.get('VV_svar')),
            'sentropy': _safe_num(tex_stats.get('VV_sent')),
            'entropy': _safe_num(tex_stats.get('VV_ent')),
            'diss': _safe_num(tex_stats.get('VV_diss')),
            'imcorr1': _safe_num(tex_stats.get('VV_imcorr1')),
            'imcorr2': _safe_num(tex_stats.get('VV_imcorr2')),
            'inertia': _safe_num(tex_stats.get('VV_inertia')),
            'shade': _safe_num(tex_stats.get('VV_shade')),
            'prom': _safe_num(tex_stats.get('VV_prom'))
        })
        intensity_dict = ee.Dictionary({
            'mu_sce': mu_sce,
            'sigma_sce': sigma_sce,
            'mu_obj': mu_obj,
            'sigma_obj': sigma_obj,
            'min_obj': min_obj,
            'max_obj': max_obj,
            'intensity_ratio': intensity_ratio,
            'isdr': isdr,
            'isri': isri
        })
        return feature.setMulti(dark_spot_dict.combine(intensity_dict).combine(texture_dict)).set('status_34f', 'ok_34f')

    def when_small_object():
        return feature.set({'status_34f': 'object_too_small', 'object_pixel_count': object_pixel_count})

    return ee.Feature(ee.Algorithms.If(valid_object, when_valid_object(), when_small_object()))


# ---------------- 单样本窗口处理：同日所有 VV Scene 拼接后再分割 ----------------
def process_feature(feature):
    feature = ee.Feature(feature)
    region = feature.geometry()
    date_str = ee.String(feature.get('Date'))
    start = ee.Date.parse('yyyyMMdd', date_str)
    end = start.advance(1, 'day')

    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(region).filterDate(start, end)
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV').sort('system:time_start'))
    image_count = s1.size()
    has_image = image_count.gt(0)

    def when_has_image():
        # 之前的逻辑：img = ee.Image(s1.first()).clip(region)，只使用当天第一景。
        # 新逻辑：拼接当天所有与窗口相交的 VV Scene，使每个像元尽可能由任一 Scene 补足。
        # 显式命名，保证覆盖率统计的字典键为 VV。
        img = s1.mosaic().clip(region).rename('VV')
        first_img = ee.Image(s1.first())  # 保持原 ERA5 匹配时刻：当天最早一景。
        scene_ids = ee.List(s1.aggregate_array('system:index'))
        scene_id = ee.String(scene_ids.join('|'))
        instrument_mode = ee.String(first_img.get('instrumentMode'))
        sar_time = ee.Date(first_img.get('system:time_start'))

        # mosaic 掩膜在窗口内的有效像元比例；用于准确标识仍存在 SAR 空洞的 patch。
        valid_dict = img.mask().reduce(ee.Reducer.min()).rename('VV').reduceRegion(
            reducer=ee.Reducer.sum(), geometry=region, scale=FEATURE_SCALE, maxPixels=1e8,
        tileScale=8
        )
        valid_pixel_count = _safe_num(
            ee.Algorithms.If(valid_dict.contains('VV'), valid_dict.get('VV'), 0), 0
        )
        total_pixel_count_sar = _safe_num(ee.Image.constant(1).rename('constant').reduceRegion(
            reducer=ee.Reducer.count(), geometry=region, scale=FEATURE_SCALE, maxPixels=1e8,
        tileScale=8
        ).get('constant'), 0)
        sar_coverage_ratio = _safe_div(valid_pixel_count, total_pixel_count_sar, 0)
        sar_missing_pixel_count = total_pixel_count_sar.subtract(valid_pixel_count).max(0)
        has_full_sar_coverage = sar_coverage_ratio.gte(SAR_COVERAGE_MIN)

        def when_missing_coverage():
            return ee.Feature(region).copyProperties(feature).set({
                SAMPLE_LABEL_PROPERTY: feature.get(SAMPLE_LABEL_PROPERTY),
                'image_count': image_count, 'scene_ids_used': scene_ids,
                'sar_coverage_ratio': sar_coverage_ratio,
                'sar_missing_pixel_count': sar_missing_pixel_count,
                'era5_count': 0, 'status': 'sar_missing_coverage',
                'status_34f': 'sar_missing_coverage'
            })

        def when_full_coverage():
            hour_start = ee.Date.fromYMD(sar_time.get('year'), sar_time.get('month'), sar_time.get('day')).advance(sar_time.get('hour'), 'hour')
            hour_end = hour_start.advance(1, 'hour')
            era5 = ee.ImageCollection('ECMWF/ERA5/HOURLY').filterBounds(region).filterDate(hour_start, hour_end).select('u_component_of_wind_10m', 'v_component_of_wind_10m')
            era5_count = era5.size()
            has_era5 = era5_count.gt(0)

            def when_has_era5():
                # 风速均值和风向 sin 均在 _wind_features 内计算一次，并随 wind_dict 写入最终 Feature。
                wind_dict = _wind_features(ee.Image(era5.first()), region, scale=1000)
                wind_mean = ee.Number(wind_dict.get('wind_mean'))
                img_filtered = lee_filter(img)
                segment_result = adaptive_threshold_window(img_filtered, region, scale=FEATURE_SCALE)
                # oil_mask 只在此处由分割结果取得一次，后续窗口级与对象级均复用该结果。
                oil_mask = ee.Image(segment_result['oil_mask'])
                oil_sum = _safe_num(oil_mask.eq(1).reduceRegion(reducer=ee.Reducer.sum(), geometry=region, scale=FEATURE_SCALE, maxPixels=1e8).get('oil_mask'), 0)
                has_oil = oil_sum.gt(0)

                def when_has_oil():
                    base_feature = build_window_feature(feature, region, img, img_filtered, oil_mask, wind_mean, scene_id, instrument_mode, sar_time, segment_result['local_mean_before'], segment_result['local_mean_after']).set({
                        'image_count': image_count, 'scene_ids_used': scene_ids,
                        'sar_coverage_ratio': sar_coverage_ratio, 'sar_missing_pixel_count': sar_missing_pixel_count,
                        'era5_count': era5_count, 'status': 'ok'
                    }).set(wind_dict)
                    return add_object_features(base_feature, region, img, oil_mask)

                def when_no_oil():
                    return ee.Feature(region).copyProperties(feature).set({
                        SAMPLE_LABEL_PROPERTY: feature.get(SAMPLE_LABEL_PROPERTY),
                        'image_count': image_count, 'scene_ids_used': scene_ids,
                        'sar_coverage_ratio': sar_coverage_ratio, 'sar_missing_pixel_count': sar_missing_pixel_count,
                        'era5_count': era5_count, 'status': 'no_oil', 'status_34f': 'no_oil'
                    })
                return ee.Feature(ee.Algorithms.If(has_oil, when_has_oil(), when_no_oil()))

            def when_no_era5():
                return ee.Feature(region).copyProperties(feature).set({
                    SAMPLE_LABEL_PROPERTY: feature.get(SAMPLE_LABEL_PROPERTY),
                    'image_count': image_count, 'scene_ids_used': scene_ids,
                    'sar_coverage_ratio': sar_coverage_ratio, 'sar_missing_pixel_count': sar_missing_pixel_count,
                    'era5_count': era5_count, 'status': 'no_era5', 'status_34f': 'no_era5'
                })
            return ee.Feature(ee.Algorithms.If(has_era5, when_has_era5(), when_no_era5()))

        return ee.Feature(ee.Algorithms.If(has_full_sar_coverage, when_full_coverage(), when_missing_coverage()))

    def when_no_image():
        return ee.Feature(region).copyProperties(feature).set({
            SAMPLE_LABEL_PROPERTY: feature.get(SAMPLE_LABEL_PROPERTY),
            'image_count': image_count, 'era5_count': 0,
            'sar_coverage_ratio': 0, 'sar_missing_pixel_count': 0,
            'status': 'no_s1', 'status_34f': 'no_s1'
        })

    return ee.Feature(ee.Algorithms.If(has_image, when_has_image(), when_no_image()))


In [6]:
# ---------------- 2) 读取全部样本窗口并批量处理 ----------------
# 这里已完成：S1 检索、ERA5 风场匹配、Lee 滤波、阈值分割、窗口统计和全部连通暗斑联合特征计算。
results_fc = fc.map(process_feature)

sample_count = fc.size()
result_count = results_fc.size()
status_hist = results_fc.aggregate_histogram('status')
status_34f_hist = results_fc.aggregate_histogram('status_34f')


In [7]:
# ---------------- 3) 导出全部样本分割结果与基础信息 ----------------
# results_fc 中包含：
# 1) 分割基础统计与窗口级 oil 特征；
# 2) status / status_34f；
# 3) 对于有效暗斑样本，已附加全部连通暗斑的联合几何、强度、背景和纹理特征。

sample_name = asset_basename(address)
export_folder = f'{SAMPLE_TAG}_info_{sample_name}'
table_prefix = f'{SAMPLE_TAG}_info_{sample_name}'

task = ee.batch.Export.table.toDrive(
    collection=results_fc,
    description=table_prefix,
    folder=export_folder,
    fileNamePrefix=table_prefix,
    fileFormat='CSV'
)

task.start()
print('Export started:', table_prefix)
print('folder:', export_folder)
print('collection: results_fc')


Export started: pos_info_pos2014
folder: pos_info_pos2014
collection: results_fc


In [8]:
# ---------------- 4) 导出可用于机器学习训练的联合暗斑特征样本 ----------------
# 不重新分割，直接从 results_fc 中筛选已得到有效联合暗斑特征的样本。
# 保留 status_34f 字段名仅为兼容原有筛选与导出流程；其值 ok_34f 现表示联合暗斑特征计算成功。

sample_name = asset_basename(address)
export_folder_34 = f'{SAMPLE_TAG}_features_{sample_name}'
EXPORT_DESC_34 = f'{SAMPLE_TAG}_features_{sample_name}'
EXPORT_PREFIX_34 = f'{SAMPLE_TAG}_features_{sample_name}'

training_fc_34 = results_fc.filter(ee.Filter.eq('status_34f', 'ok_34f'))

csv_task_34 = ee.batch.Export.table.toDrive(
    collection=training_fc_34,
    description=EXPORT_DESC_34,
    folder=export_folder_34,
    fileNamePrefix=EXPORT_PREFIX_34,
    fileFormat='CSV'
)

csv_task_34.start()
print('Joint-dark-spot feature CSV export task submitted.')
print('folder:', export_folder_34)
print('collection: training_fc_34')


Joint-dark-spot feature CSV export task submitted.
folder: pos_features_pos2014
collection: training_fc_34
